In [8]:
# ===============================================
# Structured Scraping Script for Indian Startup Funding Data (Two Parts: 2024-2025 & 2020-2022)
# ===============================================
# Overview:
# This script uses Selenium to scrape funding data from StartupTalky, split into two parts due to table structure differences:
# - Part 1 (2024-2025): 6-column tables (Company, Sector, Headquarters, Amount, Round, Investors) → 'funding_2024-2025.csv'
# - Part 2 (2020-2022): 9-column tables (Company, Founded, HQ, Sector, Description, Founders, Investors, Amount, Stage) → Mapped to same standard → 'funding_2020-2022.csv'
# Loops over URLs, extracts under H2 headings, adds Year/Month, combines per part.
# 
# Why Structured/Split? 
# - Modular: scrape_year function handles format-specific extraction.
# - Comments: Step-by-step for procces
# - Efficient: One run for all; progress prints.
# - Output: Two CSVs with ~2K rows each (standard columns: Company, Sector, Headquarters, Amount, Funding_Round_Type, Lead_Investors, Year, Month).
# 
# Requirements: pip install selenium webdriver-manager pandas
# Run: Jupyter/Colab/local (non-headless: set headless=False).
# ===============================================

# Step 1: Imports
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import re
import time

# Step 2: Define URLs by Group
urls_recent = {  # 2024-2025: 6-col format
    2024: 'https://startuptalky.com/indian-startups-funding-investor-data-2024/',
    2025: 'https://startuptalky.com/indian-startups-funding-investors-data-2025/'
}
urls_older = {   # 2020-2022: 9-col format
    2020: 'https://startuptalky.com/indian-funding-investor-data/',
    2021: 'https://startuptalky.com/indian-startup-funding-investors-data-2021/',
    2022: 'https://startuptalky.com/indian-startups-funding-investors-data-2022/'
}

# Step 3: Setup Selenium Driver
def setup_driver(headless=True):
    """Initialize Chrome driver with options."""
    options = Options()
    if headless:
        options.add_argument('--headless')  # Comment out for visible demo
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    return driver

# Step 4: Function to Scrape One Year (Handles Format Differences)
def scrape_year(driver, year, url, format_type='recent'):
    """Scrape tables for a year, extract H2 for Month, map columns based on format_type ('recent' for 6-col, 'older' for 9-col)."""
    print(f"\n--- Scraping Year {year} from {url} ({format_type} format) ---")
    driver.get(url)
    wait = WebDriverWait(driver, 10)
    
    # Find all H2 headings
    h2_headings = driver.find_elements(By.TAG_NAME, 'h2')
    print(f"Found {len(h2_headings)} H2 headings")
    
    year_dfs = []  # List to hold DFs for this year
    
    for i, h2 in enumerate(h2_headings):
        try:
            heading_text = h2.text.strip()
            print(f"  Processing H2 {i+1}: {heading_text[:50]}...")  # Truncated for log
            
            # Extract month name (general regex)
            month_match = re.search(r'- ([A-Z][a-z]+) \d{4}', heading_text)
            if month_match:
                month_name = month_match.group(1)
            else:
                # Fallback: search in brackets e.g., [01 September]
                month_match2 = re.search(r'\[(\d{2}) ([A-Z][a-z]+)', heading_text)
                month_name = month_match2.group(2) if month_match2 else 'January'
            print(f"    Month: {month_name}")
            
            # Find next table after this H2
            next_table = h2.find_element(By.XPATH, 'following::table[1]')
            
            # Extract table rows (skip header)
            rows = next_table.find_elements(By.TAG_NAME, 'tr')
            data = []
            for row in rows[1:]:
                cols = [col.text.strip() for col in row.find_elements(By.TAG_NAME, 'td')]
                if len(cols) >= 3:  # Min valid row
                    if format_type == 'recent':  # 6-col: Company(0), Sector(1), HQ(2), Amount(3), Round(4), Investors(5)
                        if len(cols) >= 6:
                            mapped_row = cols[:6]  # Direct map
                        else:
                            continue
                    else:  # 'older' 9-col: Company(0), Founded(1), HQ(2), Sector(3), Desc(4), Founders(5), Investors(6), Amount(7), Stage(8)
                        if len(cols) >= 9:
                            mapped_row = [
                                cols[0],  # Company
                                cols[3],  # Sector
                                cols[2],  # Headquarters
                                cols[7],  # Amount
                                cols[8],  # Funding_Round_Type (Stage)
                                cols[6]   # Lead_Investors
                            ]
                        else:
                            continue
                    data.append(mapped_row)
            
            if not data:
                continue
            
            # Create DF
            df = pd.DataFrame(data, columns=['Company', 'Sector', 'Headquarters', 'Amount', 'Funding_Round_Type', 'Lead_Investors'])
            df['Year'] = year
            df['Month'] = month_name
            df['Amount'] = df['Amount'].replace('Undisclosed', pd.NA)  # Basic clean
            year_dfs.append(df)
            print(f"    Added {len(data)} rows")
            time.sleep(0.1)  # Polite delay
        except Exception as e:
            print(f"    Error on H2 {i+1}: {e}")
            continue
    
    if year_dfs:
        year_df = pd.concat(year_dfs, ignore_index=True)
        print(f"Total for {year}: {year_df.shape[0]} rows")
        return year_df
    else:
        print(f"No data for {year}")
        return pd.DataFrame()  # Empty DF

# Step 5: Main Execution - Scrape Recent Years (2024-2025)
driver = setup_driver(headless=True)  # Or False for demo
recent_dfs = []  # For 2024-2025

try:
    for year, url in urls_recent.items():
        year_df = scrape_year(driver, year, url, format_type='recent')
        if not year_df.empty:
            recent_dfs.append(year_df)
        time.sleep(1)  # Delay between years
finally:
    driver.quit()  # Close after recent

# Combine Recent
if recent_dfs:
    recent_combined = pd.concat(recent_dfs, ignore_index=True)
    print(f"\n=== 2024-2025 SUMMARY ===")
    print(f"Total rows: {recent_combined.shape[0]}")
    print(recent_combined.head())
    recent_combined.to_csv('funding_2024-2025.csv', index=False)
    print("Saved 'funding_2024-2025.csv'")

# Step 6: Restart Driver & Scrape Older Years (2020-2022)
driver = setup_driver(headless=True)
older_dfs = []  # For 2020-2022

try:
    for year, url in urls_older.items():
        year_df = scrape_year(driver, year, url, format_type='older')
        if not year_df.empty:
            older_dfs.append(year_df)
        time.sleep(1)  # Delay
finally:
    driver.quit()

# Combine Older
if older_dfs:
    older_combined = pd.concat(older_dfs, ignore_index=True)
    print(f"\n=== 2020-2022 SUMMARY ===")
    print(f"Total rows: {older_combined.shape[0]}")
    print(older_combined.head())
    older_combined.to_csv('funding_2020-2022.csv', index=False)
    print("Saved 'funding_2020-2022.csv'")
else:
    print("No data scraped—check URLs/selectors")

# ===============================================
# End of Script
# "Split scraping by format (6-col recent vs. 9-col older), mapping to standard columns for consistency.
# Outputs two CSVs (~2K rows each); easy to concat later for full dataset."
# ===============================================


--- Scraping Year 2024 from https://startuptalky.com/indian-startups-funding-investor-data-2024/ (recent format) ---
Found 19 H2 headings
  Processing H2 1: Indian Startup Funding - December 2024 [23 - 28 De...
    Month: December
    Added 14 rows
  Processing H2 2: Indian Startup Funding - December 2024 [16 - 21 De...
    Month: December
    Added 16 rows
  Processing H2 3: Indian Startup Funding - December 2024 [09 - 14 De...
    Month: December
    Added 28 rows
  Processing H2 4: Indian Startup Funding - December 2024 [02 - 07 De...
    Month: December
    Added 28 rows
  Processing H2 5: Indian Startup Funding - November 2024 [25 - 30 No...
    Month: November
    Added 14 rows
  Processing H2 6: Indian Startup Funding - November 2024 [18 - 23 No...
    Month: November
    Added 19 rows
  Processing H2 7: Indian Startup Funding - November 2024 [11 - 16 No...
    Month: November
    Added 19 rows
  Processing H2 8: Indian Startup Funding - November 2024 [04 - 08 No...
    Month: 

In [9]:
# data prepare (2015-2019).

import pandas as pd

# Load the dataset
df = pd.read_csv("funding_2015-2019.csv")

# Remove first ('Sr No') and last ('Remarks') columns
df = df.iloc[:, 1:-1]

# Convert 'Date dd/mm/yyyy' column into datetime format
df['Date dd/mm/yyyy'] = pd.to_datetime(df['Date dd/mm/yyyy'], errors='coerce', dayfirst=True)

# Create new columns for Year and Month Name
df['Year'] = df['Date dd/mm/yyyy'].dt.year
df['Month'] = df['Date dd/mm/yyyy'].dt.month_name()

# Convert year to integer (removes .0)
df['Year'] = df['Year'].astype('Int64')

# Remove rows where year == 2020
df = df[df['Year'] != 2020]

# Rename columns as per your structure
df = df.rename(columns={
    'Startup Name': 'Company',
    'Industry Vertical': 'Sector',
    'City  Location': 'Headquarters',
    'Amount in USD': 'Amount',
    'InvestmentnType': 'Funding_Round_Type',
    'Investors Name': 'Lead_Investors'
})

# Reorder and keep only the required columns
df = df[['Company', 'Sector', 'Headquarters', 'Amount',
         'Funding_Round_Type', 'Lead_Investors', 'Year', 'Month']]

# Display the first few rows
print(df.head())

# Optional: Save cleaned dataset
df.to_csv("funding_2015_2019.csv", index=False)
print(" Cleaned dataset saved as 'funding_2015_2019.csv'")


         Company                        Sector Headquarters       Amount  \
7         Ecozen                    Technology         Pune    60,00,000   
8       CarDekho                    E-Commerce      Gurgaon  7,00,00,000   
9   Dhruva Space                     Aerospace    Bengaluru  5,00,00,000   
10        Rivigo                    Technology      Gurgaon  2,00,00,000   
11    Healthians  B2B-focused foodtech startup    Bengaluru  1,20,00,000   

   Funding_Round_Type                                Lead_Investors  Year  \
7            Series A                   Sathguru Catalyzer Advisors  2019   
8            Series D                   Ping An Global Voyager Fund  2019   
9                Seed                Mumbai Angels, Ravikanth Reddy  2019   
10           Series F  SAIF Partners, Spring Canter Investment Ltd.  2019   
11           Series C       Paytm, NPTK, Sabre Partners and Neoplux  2019   

       Month  
7   December  
8   December  
9   December  
10  December  
11  D

In [ ]:
# column amount setupt 2024-2025
# hear we change clean_dataset.csv to again funding_startup_data

import pandas as pd
import re
import numpy as np

# --- EXCHANGE RATES ---
USD_TO_INR = 83
SGD_TO_INR = 67

# ---------- UNIVERSAL CONVERTER ----------
def convert_to_cr(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    # Remove brackets completely
    text = re.sub(r'\(.*?\)', '', text)

    # Remove ~ + comma
    text = text.replace(",", "").replace("+", "").strip()

    # ---------- RULE 1: Rs X crore ----------
    m = re.search(r'(Rs|₹)\s*([\d\.]+)\s*(crore|Cr|CR)', text, re.IGNORECASE)
    if m:
        return float(m.group(2))  # already in crore

    # ---------- RULE 7: Rs X lakh ----------
    m = re.search(r'(Rs|₹)\s*([\d\.]+)\s*(lakh|Lakh)', text, re.IGNORECASE)
    if m:
        return float(m.group(2)) / 100  # lakh to crore

    # ---------- RULE 10: INR X crore ----------
    m = re.search(r'INR\s*([\d\.]+)\s*(crore|Cr|CR)', text, re.IGNORECASE)
    if m:
        return float(m.group(1))

    # ---------- RULE 12 SGD $X million ----------
    m = re.search(r'SGD\s*\$?\s*([\d\.]+)', text, re.IGNORECASE)
    if m:
        val = float(m.group(1)) * 1_000_000 * SGD_TO_INR
        return val / 10_000_000  # INR → crore

    # ---------- RULE 5: $X - $Y → take first ----------
    m = re.search(r'\$([\d\.]+)\s*-\s*\$?([\d\.]+)', text)
    if m:
        val = float(m.group(1)) * 1_000_000 * USD_TO_INR
        return val / 10_000_000

    # ---------- RULE 8: $500K ----------
    m = re.search(r'\$([\d\.]+)\s*K', text, re.IGNORECASE)
    if m:
        val = float(m.group(1)) * 1000 * USD_TO_INR
        return val / 10_000_000

    # ---------- RULE 9: $X Mn ----------
    m = re.search(r'\$([\d\.]+)\s*(Mn|MN)', text)
    if m:
        val = float(m.group(1)) * 1_000_000 * USD_TO_INR
        return val / 10_000_000

    # ---------- RULE 3/6: $X million ----------
    m = re.search(r'\$([\d\.]+)\s*(million|Million|M)', text)
    if m:
        val = float(m.group(1)) * 1_000_000 * USD_TO_INR
        return val / 10_000_000

    # ---------- Rule 13: ~$1 million ----------
    m = re.search(r'\$?~\s*([\d\.]+)\s*(million|M)', text)
    if m:
        val = float(m.group(1)) * 1_000_000 * USD_TO_INR
        return val / 10_000_000

    # ---------- Bare number like 25M / 0.6M ----------
    m = re.search(r'([\d\.]+)\s*M$', text)
    if m:
        val = float(m.group(1)) * 1_000_000 * USD_TO_INR
        return val / 10_000_000

    # ---------- Bare only Rs number (assume crore) ----------
    m = re.search(r'^(Rs|₹)?\s*([\d\.]+)$', text)
    if m:
        return float(m.group(2))

    return np.nan


# ---------- LOAD DATA ----------
df = pd.read_csv("funding_2024-2025.csv")

# ---------- APPLY MAIN CONVERSION ----------
df["Amount"] = df["Amount"].apply(convert_to_cr)


# ---------- EXTERNAL AMOUNTS TO ADD (254–270) ----------
external_amounts = {
    254: "$0.7 million",
    255: "$10 million",
    256: "$20 million",
    257: "$2 million",
    258: "$1.1 million",
    259: "$4 million",
    260: "$1 million",
    261: "$1.6 million",
    262: "$3 million",
    263: "0.8M",
    264: "25M",
    265: "3.5M",
    266: "27M",
    267: "12M",
    268: "9M",
    269: "2.3M",
    270: "1.2M"
}

# Convert and inject them into Amount_Cr
for idx, val in external_amounts.items():
    df.loc[idx, "Amount"] = round(convert_to_cr(val), 2)

# ---------- SAVE NEW DATASET ----------
df.to_csv("clean_dataset.csv", index=False)

print(" Conversion complete! Saved as clean_dataset.csv")


In [ ]:
# hear i change clean_dataset.csv to funding_2024-2025

In [68]:
# merge 3 dataset

import pandas as pd

# Load your 3 datasets
df1 = pd.read_csv("funding_2015_2019.csv")
df2 = pd.read_csv("funding_2020-2022.csv")
df3 = pd.read_csv("funding_2024-2025.csv")

# Combine all datasets
merged_df = pd.concat([df1, df2, df3], ignore_index=True)

# Optional: drop duplicate rows (if any)
merged_df.drop_duplicates(inplace=True)

# Check shape and preview
print(" Merged dataset shape:", merged_df.shape)
print(merged_df.head())

# Save the merged dataset
merged_df.to_csv("merged_startup_data.csv", index=False)
print(" Saved as 'merged_startup_data.csv'")


✅ Merged dataset shape: (7293, 8)
        Company                        Sector Headquarters       Amount  \
0        Ecozen                    Technology         Pune    60,00,000   
1      CarDekho                    E-Commerce      Gurgaon  7,00,00,000   
2  Dhruva Space                     Aerospace    Bengaluru  5,00,00,000   
3        Rivigo                    Technology      Gurgaon  2,00,00,000   
4    Healthians  B2B-focused foodtech startup    Bengaluru  1,20,00,000   

  Funding_Round_Type                                Lead_Investors  Year  \
0           Series A                   Sathguru Catalyzer Advisors  2019   
1           Series D                   Ping An Global Voyager Fund  2019   
2               Seed                Mumbai Angels, Ravikanth Reddy  2019   
3           Series F  SAIF Partners, Spring Canter Investment Ltd.  2019   
4           Series C       Paytm, NPTK, Sabre Partners and Neoplux  2019   

      Month  
0  December  
1  December  
2  December  
3 

In [17]:
!pip install rapidfuzz


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------------- ------------------- 0.8/1.5 MB 2.7 MB/s eta 0:00:01
   ---------------------------------- ----- 1.3/1.5 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 2.4 MB/s eta 0:00:00


In [14]:
# data cleaing

import pandas as pd
import numpy as np
import re

# Consolidated Cleaning Pipeline: Loads 'merged_startup_data.csv' and applies all steps sequentially
# Preserves original row order throughout (handles sorting in HQ cleaning by restoring index)

# === STEP 0: Load Original Data ===
df = pd.read_csv("merged_startup_data.csv")
print(f"Loaded original dataset: {len(df)} rows, {len(df.columns)} columns")
original_index = df.index  # Preserve for order restoration

# === STEP 1: Basic Cleaning (from first script) ===
df_raw = df.copy()

# Normalize column names
df.columns = df.columns.str.strip().str.replace(r'\s+', ' ', regex=True)

# Trim whitespace in string columns and replace empties with NaN
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'': np.nan, 'nan': np.nan, 'None': np.nan})

# Drop exact duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Dropped exact duplicates: {before - len(df)}")

# Missing counts for critical fields
print("Missing counts (Company, Year, Month):")
print(df[['Company','Year','Month']].isnull().sum())

# Fill non-critical text columns with 'Unknown'
text_fill_cols = ['Sector', 'Headquarters', 'Funding_Round_Type', 'Lead_Investors']
for c in text_fill_cols:
    if c in df.columns:
        df[c] = df[c].fillna('Unknown')

# Clean Amount to INR numeric (basic version)
def amount_to_inr(x):
    if pd.isna(x):
        return 0.0
    s = str(x).lower().replace(',', '').strip()
    if s in ['', 'nan', 'undisclosed', 'unknown', 'n/a', '-']:
        return 0.0
    m = re.search(r'(\d+(?:\.\d+)?)', s)
    if not m:
        return 0.0
    val = float(m.group(1))
    if re.search(r'\bk\b', s):
        val *= 1e3
    elif re.search(r'\bm\b', s):
        val *= 1e6
    elif re.search(r'\bb\b', s):
        val *= 1e9
    if 'usd' in s or '$' in s:
        val *= 83.0
    return round(val, 2)

if 'Amount' in df.columns:
    df['Amount'] = df['Amount'].apply(amount_to_inr)
    df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce').fillna(0.0)
else:
    print("Warning: 'Amount' column not found.")

# Ensure Year and Month consistency
if 'Year' in df.columns:
    df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
else:
    print("Warning: 'Year' column not found.")

if 'Month' in df.columns:
    df['Month'] = df['Month'].astype(str).str.strip().replace({'nan': np.nan})
    valid_months = ['January','February','March','April','May','June',
                    'July','August','September','October','November','December']
    df.loc[~df['Month'].isin(valid_months) & df['Month'].notnull(), 'Month'] = 'Various'
else:
    print("Warning: 'Month' column not found.")

# Drop rows missing Company
critical_before = len(df)
df = df.dropna(subset=['Company'])
print(f"Dropped rows with missing Company: {critical_before - len(df)}")

# Restore original relative order after drops (approximate; assumes drops don't disrupt much)
df = df.reset_index(drop=True)

print("\nBasic cleaning done.")

# === STEP 2: Company Cleaning ===
def normalize_company_name(name):
    if pd.isna(name):
        return np.nan
    name = str(name).strip()
    name = name.replace('\xe2\x80\x99', "'")
    name = name.replace("\\'", "'")
    name = name.replace('\\"', '"')
    name = name.lower()
    name = re.sub(r'[^\w\s]', '', name)
    name = re.sub(r'\s+', ' ', name)
    suffixes = [
        'private limited', 'pvt ltd', 'pvt', 'limited', 'ltd', 'inc', 
        'llp', 'l l p', 'co', 'company', 'corp', 'corporation', 'india', 
        'technologies', 'solutions', 'services', 'group', 'holdings', 'llc', 'gmbh'
    ]
    for s in suffixes:
        name = re.sub(r'\b' + re.escape(s) + r'\b', '', name, flags=re.IGNORECASE)
    name = name.strip()
    name = name.title()
    return name

df['Company_Cleaned'] = df['Company'].apply(normalize_company_name)

mapping = {
    'Flipkart Pvt': 'Flipkart', 'Flipkart India': 'Flipkart', 'Flipkart Pvt Ltd': 'Flipkart',
    'Oyo Rooms': 'Oyo', 'Oyo Hotels': 'Oyo', 'Oyo Hotels Homes': 'Oyo',
    'Paytm Payments Bank': 'Paytm', 'Paytm Mall': 'Paytm', 'Paytm Money': 'Paytm',
    'Ola Cabs': 'Ola', 'Ola Electric': 'Ola', 'Zomato Media': 'Zomato', 'Zomato India': 'Zomato',
    'Swiggy Delivery': 'Swiggy', 'Swiggy India': 'Swiggy', 'Policybazaar': 'Policy Bazaar',
    'Nykaa Fashion': 'Nykaa', 'Nykaa E Retail': 'Nykaa', 'Delhivery Pvt': 'Delhivery',
    'Delhivery Logistics': 'Delhivery', 'Meesho India': 'Meesho', 'Lenskart Solutions': 'Lenskart',
    'Lenskart Pvt': 'Lenskart', 'Lenskart Com': 'Lenskart', 'Dream11 Fantasy': 'Dream11',
    'Curefit': 'Cure Fit', 'Curefit Healthcare': 'Cure Fit',
    'BYJU\'S': 'Byjus', '"BYJU\'S"': 'Byjus', 'Byju\'s': 'Byjus', 'BYJU’S': 'Byjus',
    'Byju’s': 'Byjus', 'Byju': 'Byjus', 'BYJU S': 'Byjus', 'Bytelearn': 'Bytelearn',
    'Prepbytes': 'Prepbytes', 'Newsbytes': 'Newsbytes', 'Bytexl': 'Bytexl', 'Bytes': 'Bytes',
    'Biryani By Kilo': 'Biryani By Kilo', 'Mobycy': 'Mobycy', 'Babyonboard': 'Babyonboard',
    'Kloseby': 'Kloseby', 'Babychakra': 'Babychakra', 'Baby Chakra': 'Babychakra',
    'BYG': 'Byg', 'Babygogo': 'Babygogo', 'Baby Berry': 'Baby Berry', 'Joy By Nature': 'Joy By Nature',
    'Mybyk': 'Mybyk', 'Bytebeam': 'Bytebeam', 'All Things Baby': 'All Things Baby',
    'Millenium Babycare': 'Millenium Babycare', 'Babynama': 'Babynama',
    'Cardekho': 'Car Dekho', 'Healthians B2B Focused Foodtech Startup': 'Healthians',
    'Aye Finance': 'Aye Finance', 'Vogo Automotive Pvt Ltd': 'Vogo Automotive',
    'No Broker': 'No Broker', 'Bira91': 'Bira91', 'Fabhotels': 'Fab Hotels',
    'Bharatpe': 'Bharat Pe', 'Blackbuck': 'Black Buck', 'Ather Energy': 'Ather Energy',
    'Kuvera': 'Kuvera', 'Medlife': 'Medlife', 'Unacademy': 'Unacademy',
    'Clever Tap': 'Clever Tap', 'Krazy Bee': 'Krazy Bee', 'Shuttl': 'Shuttl',
    'Increff': 'Increff', 'Zilingo': 'Zilingo', 'Vyome Therapeutics Inc': 'Vyome Therapeutics',
    'Samunnati Financial Intermediation Services Pvt Ltd': 'Samunnati Financial',
    'Urban Clap Technologies Pvt Ltd': 'Urban Clap', 'Guiddoo': 'Guiddoo',
    'Career Anna': 'Career Anna', 'Nagpur Wholesale': 'Nagpur Wholesale',
    'Shopkirana': 'Shop Kirana', 'Build Supply': 'Build Supply', 'Go Desi': 'Go Desi',
    'Veritas Finance Ltd': 'Veritas Finance', 'Meesho': 'Meesho',
    'Mobile Premier League': 'Mobile Premier League', 'A R Bon Vivants': 'A R Bon Vivants',
    'Milk Basket': 'Milk Basket', 'Drive U': 'Drive U', 'Cleanse Car': 'Cleanse Car',
    'Automation Anywhere': 'Automation Anywhere', 'Healthify Me': 'Healthify Me',
    'Genius Corner': 'Genius Corner', 'Aavishkaar Intellecap Group': 'Aavishkaar Intellecap',
    'Skillbox': 'Skillbox', 'Signzy': 'Signzy', 'Engineer Ai': 'Engineer Ai',
    'In Cred Finance': 'In Cred', 'Roposo': 'Roposo', 'Northmist': 'Northmist',
    'Origo Commodities India Pvt Ltd': 'Origo Commodities', 'Grover Zampa': 'Grover Zampa'
}

mapping_clean = {normalize_company_name(k): v.title() for k, v in mapping.items()}
df['Company_Cleaned'] = df['Company_Cleaned'].replace(mapping_clean)
df['Company_Cleaned'] = df['Company_Cleaned'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

print(f"Company cleaning: Unique before {df_raw['Company'].nunique()}, after {df['Company_Cleaned'].nunique()}")

# === STEP 3: Headquarters Cleaning ===
hq_col = 'Headquarters'
df[hq_col] = df[hq_col].astype(str).str.strip().str.lower()
df[hq_col] = df[hq_col].replace(['-', 'nan', 'none', 'na', 'null'], np.nan)
df[hq_col] = df[hq_col].apply(lambda x: re.sub(r'\\xc2\\xa0', '', str(x)).strip())
df[hq_col] = df[hq_col].apply(lambda x: x.split('/')[0].split(',')[0].strip() if isinstance(x, str) else x)
df[hq_col] = df[hq_col].apply(
    lambda x: np.nan if len(str(x).split()) > 4 or re.search(r'services|ventures|protocol|product|finance', str(x)) else x
)

city_map = {
    'bangaluru': 'bangalore', 'banglore': 'bangalore', 'bengalaru': 'bangalore', 'benglaru': 'bangalore',
    'bengaluru': 'bangalore', 'gurugram': 'gurgaon', 'new delhi': 'delhi', 'newdelhi': 'delhi',
    'delhi / ncr': 'delhi', 'ahmadabad': 'ahmedabad', 'ahemdabad': 'ahmedabad', 'nasik': 'nashik',
    'bhubaneshwar': 'bhubaneswar', 'bhubhaneshwar': 'bhubaneswar', 'bhubhneshwar': 'bhubaneswar',
    'trivandrum': 'thiruvananthapuram', 'tirivanthapuram': 'thiruvananthapuram', 'california': 'usa',
    'san franciscao': 'san francisco', 'san francisco bay area': 'san francisco',
    'san francisco, california': 'san francisco', 'san jose,': 'san jose', 'mumba': 'mumbai'
}
df[hq_col] = df[hq_col].replace(city_map)

# Extract money from HQ to Amount
money_pattern = r'^\$?\d+(\.\d+)?\s?(m|million)?$'
mask_money = df[hq_col].str.contains(money_pattern, case=False, na=False)
df.loc[mask_money, 'Amount'] = df.loc[mask_money, hq_col].apply(
    lambda x: float(re.sub(r'[^\d\.]', '', x)) * 1_000_000 if 'm' in str(x).lower() else float(re.sub(r'[^\d\.]', '', x))
)
df.loc[mask_money, hq_col] = np.nan

# Ffill/Bfill per company WITHOUT sorting the main df (preserve order)
# Create a temp sorted df for filling, then map back
df_temp = df[['Company', 'Year', hq_col]].copy()  # Include 'Year' here
df_temp_sorted = df_temp.sort_values(['Company', 'Year'])
df_temp_sorted[hq_col] = df_temp_sorted.groupby('Company')[hq_col].transform(lambda x: x.ffill().bfill())
df[hq_col] = df_temp_sorted[hq_col].reindex(df.index).values  # Map back to original order

# Final HQ cleanup
df[hq_col] = df[hq_col].apply(lambda x: re.sub(r'[^a-z\s]', '', x) if isinstance(x, str) else x)
df[hq_col] = df[hq_col].str.strip().str.replace(r'\s+', ' ', regex=True)
df[hq_col] = df[hq_col].str.title()

print("Headquarters cleaning complete.")
print(f"Unique HQs: {df[hq_col].nunique()}")

# === STEP 4: Amount Cleaning (to Crore) ===

import re
import pandas as pd
import numpy as np

def clean_amount(value, year):
    #  If 2024 or 2025 → return raw value (no conversion)
    if year in [2024, 2025]:
        try:
            return float(value)
        except:
            return value

    if pd.isna(value):
        return 0

    s = str(value).strip().lower()
    s = s.replace(',', '').replace('–', '').replace('-', '').replace('—', '').strip()

    if s in ['', 'na', 'n/a', 'none', 'null', 'unknown', 'undisclosed', '$undisclosed', '\xc2\xa0n/a']:
        return 0

    if not any(char.isdigit() for char in s):
        return 0

    #  NEW: Remove brackets (Rs xx crore) or ($yy million)
    s = re.sub(r'\(.*?\)', '', s).strip()

    #  NEW: handle Rs crore directly
    m = re.search(r'(rs|₹)\s*([\d\.]+)\s*(crore|cr)', s)
    if m:
        return round(float(m.group(2)), 2)

    #  NEW: handle INR crore
    m = re.search(r'inr\s*([\d\.]+)\s*(crore|cr)', s)
    if m:
        return round(float(m.group(1)), 2)

    #  NEW: Rs lakh → crore
    m = re.search(r'(rs|₹)\s*([\d\.]+)\s*lakh', s)
    if m:
        return round(float(m.group(2)) / 100, 2)

    #  NEW: $500-$550 → take first
    m = re.search(r'\$([\d\.]+)\s*-\s*\$?([\d\.]+)', s)
    if m:
        first = float(m.group(1))
        conv = first * 1_000_000 * 83 / 10_000_000
        return round(conv, 2)

    #  NEW: USD million
    m = re.search(r'\$([\d\.]+)\s*(million|m)', s)
    if m:
        v = float(m.group(1))
        conv = v * 1_000_000 * 83 / 10_000_000
        return round(conv, 2)

    #  NEW: ~$1 million
    m = re.search(r'~\s*\$?([\d\.]+)\s*(million|m)', s)
    if m:
        v = float(m.group(1))
        conv = v * 1_000_000 * 83 / 10_000_000
        return round(conv, 2)

    #  NEW: $500K
    m = re.search(r'\$([\d\.]+)\s*k', s)
    if m:
        v = float(m.group(1))
        conv = v * 1000 * 83 / 10_000_000
        return round(conv, 2)

    #  NEW: SGD $50 million
    m = re.search(r'sgd\s*\$?([\d\.]+)', s)
    if m:
        v = float(m.group(1))
        conv = v * 1_000_000 * 67 / 10_000_000
        return round(conv, 2)

    #  Continue your original logic below
    multiplier = 1
    if 'k' in s:
        multiplier = 1_000
    elif 'm' in s or 'mn' in s or 'million' in s:
        multiplier = 1_000_000
    elif 'b' in s or 'billion' in s:
        multiplier = 1_000_000_000
    elif 'cr' in s or 'crore' in s:
        multiplier = 10_000_000

    is_usd = ('$' in s) or ('usd' in s) or (2015 <= year <= 2019)

    s = re.sub(r'$[^)]*$', '', s)
    s = re.sub(r'[^\d.]', '', s)

    num_match = re.search(r'\d+(.\d+)?', s)
    if not num_match:
        return 0

    try:
        amount = float(num_match.group()) * multiplier
    except:
        return 0

    if is_usd:
        amount *= 83

    amount_in_cr = amount / 10_000_000
    return round(amount_in_cr, 2)


#  Apply
df['Amount_Cr'] = df.apply(lambda x: clean_amount(x['Amount'], x['Year']), axis=1)
print(f"Amount_Cr stats: {df['Amount_Cr'].describe()}")


# === STEP 5: Funding_Round_Type Cleaning ===
col = 'Funding_Round_Type'
df[col] = df[col].astype(str).str.lower().str.strip().replace(['-', '–', '—', 'none', 'nan', 'unknown', 'undisclosed', 'undisclose', 'unattributed', 'undiclosed'], 'unknown')
df[col] = df[col].apply(lambda x: re.sub(r'[^a-z0-9\s\+\-]', ' ', x))
df[col] = df[col].str.replace(r'\s+', ' ', regex=True).str.strip()

funding_map = {
    'seed': 'Seed', 'seed funding': 'Seed', 'seed round': 'Seed', 'seed/angel funding': 'Seed',
    'seed / angel funding': 'Seed', 'seed/ angel funding': 'Seed', 'seed / angle funding': 'Seed',
    'seed+': 'Seed', 'seed investment': 'Seed', 'early seed': 'Seed', 'post-seed': 'Seed',
    'seed a': 'Seed', 'seed fund': 'Seed', 'seed funding round': 'Seed', 'seeds': 'Seed',
    'pre seed': 'Pre-Seed', 'pre-seed': 'Pre-Seed', 'preseed': 'Pre-Seed', 'pre seed round': 'Pre-Seed',
    'pre-seed funding': 'Pre-Seed', 'pre-seed round': 'Pre-Seed',
    'pre series a': 'Pre-Series A', 'pre-series a': 'Pre-Series A', 'pre-series a round': 'Pre-Series A',
    'pre-series a1': 'Pre-Series A', 'pre series a1': 'Pre-Series A',
    'series a': 'Series A', 'seriesa': 'Series A', 'series a1': 'Series A', 'series a+ debt funding': 'Series A',
    'series a round': 'Series A', 'extended series a': 'Series A', 'series a2': 'Series A',
    'series a3': 'Series A', 'series a+': 'Series A', 'seriesaa': 'Series A', 'series a-1': 'Series A',
    'series b': 'Series B', 'seriesb': 'Series B', 'series b1': 'Series B', 'series b2': 'Series B',
    'series b3': 'Series B', 'extended series b': 'Series B', 'series b round': 'Series B',
    'series c': 'Series C', 'seriesc': 'Series C', 'series c round': 'Series C', 'series c extension': 'Series C',
    'series d': 'Series D', 'seriesd': 'Series D', 'series d1': 'Series D', 'series d pre ipo funding round': 'Series D',
    'series d round': 'Series D', 'series e': 'Series E', 'series e round': 'Series E', 'series e2': 'Series E',
    'series f': 'Series F', 'seriesf': 'Series F', 'series f1': 'Series F', 'series f2': 'Series F',
    'series g': 'Series G', 'seriesg': 'Series G', 'series g round': 'Series G',
    'series h': 'Series H', 'seriesh': 'Series H', 'series i': 'Series I', 'series j': 'Series J',
    'private equity': 'Private Equity', 'privateequity': 'Private Equity', 'private equity round': 'Private Equity',
    'pe': 'Private Equity', 'debt': 'Debt', 'debt funding': 'Debt', 'debt financing': 'Debt',
    'debt round': 'Debt', 'conventional debt': 'Debt', 'structured debt': 'Debt', 'convertible debt': 'Debt',
    'debt fund': 'Debt', 'debt-funding': 'Debt', 'debt and equity round': 'Debt + Equity',
    'debt and preference capital': 'Debt + Equity', 'equity round': 'Equity', 'equity funding': 'Equity',
    'equity and debt funding round': 'Debt + Equity', 'equity financing': 'Equity', 'equity investment': 'Equity',
    'equity capital': 'Equity', 'equity funding round': 'Equity', 'strategic investment': 'Strategic Investment',
    'strategic funding round': 'Strategic Investment', 'strategic round': 'Strategic Investment',
    'acquisition': 'Acquisition', 'strategic stake acquisition': 'Acquisition', 'angel': 'Angel',
    'angel round': 'Angel', 'angel funding': 'Angel', 'angel fund': 'Angel', 'angel / seed funding': 'Angel',
    'venture round': 'Venture', 'venture funding': 'Venture', 'venture debt': 'Venture Debt', 'venture': 'Venture',
    'bridge': 'Bridge', 'bridge round': 'Bridge', 'bridge funding round': 'Bridge', 'ipo': 'IPO',
    'post ipo': 'IPO', 'pre ipo': 'IPO', 'pre-ipo funding': 'IPO', 'pre-ipo funding round': 'IPO',
    'pre ipo round': 'IPO', 'growth funding': 'Growth', 'growth funding round': 'Growth',
    'maiden funding round': 'Initial Funding', 'first funding round': 'Initial Funding',
    'funding round': 'Unspecified Round', 'new funding round': 'Unspecified Round', 'unknown': 'Unknown'
}
df[col] = df[col].map(funding_map).fillna('Other')
print("Funding_Round_Type cleaning: Unique values:")
print(df[col].value_counts().head(10))

# === STEP 6: Lead_Investors Cleaning ===
df['Lead_Investors'] = df['Lead_Investors'].fillna('Unknown')
df['Lead_Investors'] = df['Lead_Investors'].replace(['-', '–', '—', 'Undisclosed', 'unknown', 'na', 'n/a'], 'Unknown', regex=True)
df['Lead_Investors'] = df['Lead_Investors'].str.strip()

investor_map = {
    'sequoia capital india': 'Sequoia Capital', 'sequoia': 'Sequoia Capital', 'sequoia capital': 'Sequoia Capital',
    'accel partners': 'Accel', 'accel india': 'Accel', 'accel': 'Accel',
    'matrix partners india': 'Matrix Partners', 'matrix partners': 'Matrix Partners',
    'softbank vision fund': 'SoftBank', 'softbank group': 'SoftBank', 'softbank': 'SoftBank',
    'tiger global management': 'Tiger Global', 'tiger global': 'Tiger Global',
    'lightspeed venture partners': 'Lightspeed', 'lightspeed india partners': 'Lightspeed', 'lightspeed': 'Lightspeed',
    'blume ventures': 'Blume Ventures', 'blume': 'Blume Ventures',
    'nexus venture partners': 'Nexus Venture Partners', 'nexus ventures': 'Nexus Venture Partners'
}

def clean_investor_name(name):
    if not isinstance(name, str) or name.strip() == '':
        return 'Unknown'
    n = re.sub(r'[^a-zA-Z0-9 ]', '', name).lower().strip()
    return investor_map.get(n, name.title().strip())

def normalize_investor_column(value):
    names = [x.strip() for x in re.split(',|&', str(value)) if x.strip() != '']
    cleaned = [clean_investor_name(x) for x in names]
    return ', '.join(cleaned) if cleaned else 'Unknown'

df['Lead_Investors'] = df['Lead_Investors'].apply(normalize_investor_column)
print("Lead_Investors cleaning: Top unique values:")
print(df['Lead_Investors'].value_counts().head(10))

# === STEP 7: Sector Cleaning ===
df['Sector_Raw'] = df['Sector'].copy()

def normalize_sector(sector):
    if pd.isna(sector):
        return np.nan
    sector = str(sector).strip()
    sector = sector.replace('\\xc2\\xa0', ' ').replace('\\xe2\\x80\\x99', "'")
    sector = sector.lower()
    sector = re.sub(r'[^\w\s&>:/]', '', sector)
    sector = re.sub(r'\s+', ' ', sector)
    suffixes = [
        'startup', 'tech', 'platform', 'company', 'app', 'brand', 'service', 'services', 
        'solutions', 'provider', 'marketplace', 'chain', 'network', 'group', 'inc', 'ltd'
    ]
    for s in suffixes:
        sector = re.sub(r'\b' + re.escape(s) + r'(s?)\b', '', sector, flags=re.IGNORECASE)
    sector = sector.strip()
    sector = sector.title()
    return sector

df['Sector'] = df['Sector'].apply(normalize_sector)

mapping = {
    # Fintech variants
    'Fintech': 'Fintech',
    'Financial Services': 'Fintech',
    'Finance': 'Fintech',
    'Financial Services > Consumer And Sme Loans': 'Fintech',
    'Fin Tech': 'Fintech',
    'Fin-Tech': 'Fintech',
    'Financial&Services': 'Fintech',
    'Financial Sevices': 'Fintech',
    'Financial Services > Banking Tech': 'Fintech',
    'Financial Services > Investment Tech': 'Fintech',
    'Financial Services > Alternative Lending': 'Fintech',
    'Finance Services': 'Fintech',
    'Financial Tech': 'Fintech',
    'Financial Service': 'Fintech',
    'Fintech > Banking Tech': 'Fintech',
    'Fintech > Investment Tech': 'Fintech',
    'Fintech > Alternative Lending': 'Fintech',
    'Fintech Platform': 'Fintech',
    'Fintech - Nbfc Lending': 'Fintech',
    'Fintech - Cross-Border Payments': 'Fintech',
    'Fintech - Payments Infrastructure': 'Fintech',
    'Fintech - Lending Platform': 'Fintech',
    'Enterprise Fintech': 'Fintech',
    'B2b Fintech': 'Fintech',
    'Api Banking / Fintech': 'Fintech',
    'Edtech/Fintech': 'Fintech',
    'Wealthtech': 'Fintech',
    'Insuretech': 'Fintech',
    'Insurtech': 'Fintech',
    'Insurance Technology': 'Fintech',
    'Nbfc': 'Fintech',
    'Nbfc - Education Loans': 'Fintech',
    'Nbfc - Msme Lending': 'Fintech',
    'Nbfc - Sme Lending': 'Fintech',
    'Nbfc-Focused Fintech': 'Fintech',
    'Msme-Focused Nbfc': 'Fintech',
    'Msme-Focused Fintech': 'Fintech',
    
    # Ecommerce variants
    'Ecommerce': 'Ecommerce',
    'E-Commerce': 'Ecommerce',
    'E Commerce': 'Ecommerce',
    'E-Commerce Of Fuel': 'Ecommerce',
    'Ecommerce Brands’ Full Service Agency': 'Ecommerce',
    'Ecommerce Logistics': 'Ecommerce',
    'Ecommerce Product Recommendation Platform': 'Ecommerce',
    'Ecommerce Delivery Locker Services': 'Ecommerce',
    'Ecommerce Marketing Software Platform': 'Ecommerce',
    'Ecommerce Website Creation Saas Platform': 'Ecommerce',
    'Ecommerce Returns Etailer': 'Ecommerce',
    'Ecommerce Data Analytics Platform': 'Ecommerce',
    'E-Commerce & M-Commerce Platform': 'Ecommerce',
    'E-Commerce Platform Solutions': 'Ecommerce',
    'E-Commerce Marketing': 'Ecommerce',
    'E-Commerce Marketing & Analytics': 'Ecommerce',
    'E-Tail': 'Ecommerce',
    'E-Tailor': 'Ecommerce',
    'Estore': 'Ecommerce',
    'E-Marketplace': 'Ecommerce',
    'E-Market': 'Ecommerce',
    'E-Commerce Logistics': 'Ecommerce',
    
    # Edtech variants
    'Edtech': 'Edtech',
    'Ed Tech': 'Edtech',
    'Ed-Tech': 'Edtech',
    'Education': 'Edtech',
    'E-Learning': 'Edtech',
    'Education Management': 'Edtech',
    'Online Education': 'Edtech',
    'Online Education Platform': 'Edtech',
    'Online Education Information Platform': 'Edtech',
    'Edtech - Schools': 'Edtech',
    'Edtech - Student Mobility': 'Edtech',
    'Edtech - Teacher Training': 'Edtech',
    'Edtech Startup': 'Edtech',
    'Edttech': 'Edtech',
    'Ed-Tech Platform': 'Edtech',
    'Edu Tech': 'Edtech',
    'Edu-Wealth': 'Edtech',
    'E-Learning Providers': 'Edtech',
    'E-Learning Service Provider': 'Edtech',
    'E-Learnig': 'Edtech',
    
    # Health variants
    'Healthcare': 'Healthcare',
    'Health Care': 'Healthcare',
    'Healthtech': 'Healthcare',
    'Health Tech': 'Healthcare',
    'Health-Tech': 'Healthcare',
    'Health & Wellness': 'Healthcare',
    'Health And Wellness': 'Healthcare',
    'Health, Wellness & Fitness': 'Healthcare',
    'Health & Fitness App': 'Healthcare',
    'Health-Tech Platform': 'Healthcare',
    'Health, Wellness': 'Healthcare',
    'Health,Wellness': 'Healthcare',
    'Healthcare,Wellness': 'Healthcare',
    'Healthcare/Edtech': 'Healthcare',
    'Healthcare > Hospital Chains': 'Healthcare',
    'Healthcare > Clinic Chains': 'Healthcare',
    'Healthcare Ai': 'Healthcare',
    'Healthcare - Employee Benefits': 'Healthcare',
    'Healthcare Payments': 'Healthcare',
    'Healthcare Booking Platforms': 'Healthcare',
    'Healthcare It Solutions & Services': 'Healthcare',
    'Healthcare Mobile App': 'Healthcare',
    'Healthcare Services Discovery Platform': 'Healthcare',
    'Healthcare Consulting Platform': 'Healthcare',
    'Healthcare Products': 'Healthcare',
    'Health Mobile App': 'Healthcare',
    'Health Insurance': 'Healthcare',
    'Health Supplements': 'Healthcare',
    'Health And Beauty Services Marketplace': 'Healthcare',
    'Health And Wellness Platform': 'Healthcare',
    'Health And Wellness Services App': 'Healthcare',
    'Health & Nutrition': 'Healthcare',
    'Health Service': 'Healthcare',
    'Healthtech - Ai-Driven Solutions': 'Healthcare',
    'Healthtech > Disease Self Management': 'Healthcare',
    'Healthtech > Fitness & Wellness Tech': 'Healthcare',
    'Healthtech > Healthcare Booking Platforms': 'Healthcare',
    
    # Food variants
    'Food & Beverages': 'Food',
    'Food And Beverages': 'Food',
    'Food & Beverage': 'Food',
    'Food And Beverage': 'Food',
    'Foodtech': 'Food',
    'Food Tech': 'Food',
    'Food-Tech': 'Food',
    'Food & Bevarages': 'Food',
    'Food&Beverages': 'Food',
    'Food &Bevrages': 'Food',
    'Food And Bevrages': 'Food',
    'Food Delivery': 'Food',
    'Food Delivery Platform': 'Food',
    'Foodtech & Logistics': 'Food',
    'Food And Agriculture > Food & Beverage Products': 'Food',
    'Food And Agriculture Tech > Food Tech': 'Food',
    'Food And Agriculture Tech > Online Grocery': 'Food',
    'Food And Agriculture > Food Service Chains': 'Food',
    'Food Service Chains': 'Food',
    'Food Processing': 'Food',
    'Food Production': 'Food',
    'Food Industry': 'Food',
    'Food Packaging': 'Food',
    'Food Startup': 'Food',
    
    # Tech/Software variants
    'Technology': 'Technology',
    'Tech': 'Technology',
    'Tech Startup': 'Technology',
    'Tech Company': 'Technology',
    'Tech Platform': 'Technology',
    'Tech Solutions': 'Technology',
    'Tech Hub': 'Technology',
    
    # AI variants
    'Ai': 'AI',
    'Artificial Intelligence': 'AI',
    'Artificial Intelligence (Ai)': 'AI',
    'Artificial Intelligence Ai': 'AI',
    'Artificial I Ntelligence': 'AI',
    'Artificial Intelligence Ecommerce Chatbot': 'AI',
    'Ai Startup': 'AI',
    'Ai Company': 'AI',
    'Ai Platform': 'AI',
    'Ai For Finance, Accounting': 'AI',
    'Ai-Driven Robotics': 'AI',
    'Ai - Workplace Solutions': 'AI',
    'Ai-Driven B2b Sales Platform': 'AI',
    'Ai For Smart City Solutions': 'AI',
    'Ai Innovation': 'AI',
    'Ai Software': 'AI',
    'Ai Security': 'AI',
    'Ai Construction Industry': 'AI',
    'Ai Business Transformation': 'AI',
    'Ai Travel Agent': 'AI',
    'Ai Customer Feedback Platform': 'AI',
    'Ai Revenue Cycle Management': 'AI',
    'Ai Assistants': 'AI',
    'Ai-Powered Adas For Two-Wheelers - Mobilitytech': 'AI',
    'Ai Chips - Semiconductor': 'AI',
    'Ai Deployment & Scaling': 'AI',
    'Ai Hiring Platform': 'AI',
    'Genai Startup': 'AI',
    'Ai-Powered Visual Merchandising - Autotech': 'AI',
    'Ai-Powered Supply Chain': 'AI',
    'Ai & Edge Computing Saas': 'AI',
    'Ai Optimisation': 'AI',
    
    # Other common categories
    'Logistics': 'Logistics',
    'Logistics & Supply Chain': 'Logistics',
    'Logistics Tech': 'Logistics',
    'Logistics Tech Platform': 'Logistics',
    'Logistics Automation': 'Logistics',
    'Logistics Service Provider': 'Logistics',
    'Logistics Intelligence': 'Logistics',
    'Logistics Solution Provider': 'Logistics',
    'Logistics Services Provider': 'Logistics',
    'Logistics - Hyperlocal Deliveries': 'Logistics',
    'Logistics & Warehousing': 'Logistics',
    'Logistics-Tech': 'Logistics',
    'Consumer > Logistics Tech': 'Logistics',
    
    'SaaS': 'SaaS',
    'Saas': 'SaaS',
    'Saas Startup': 'SaaS',
    'Saas Platform': 'SaaS',
    'SaaS Ai Platform': 'SaaS',
    'SaaS - Research Analytics': 'SaaS',
    'SaaS - Cloud Security': 'SaaS',
    'SaaS - Physical Therapy': 'SaaS',
    'SaaS/Edtech': 'SaaS',
    'SaaS, Ecommerce': 'SaaS',
    
    'Agritech': 'Agritech',
    'Agri Tech': 'Agritech',
    'Agri-Tech': 'Agritech',
    'Agriculture': 'Agritech',
    'Agri-Commerce': 'Agritech',
    'Agri-Financing': 'Agritech',
    'Agri-Biotech': 'Agritech',
    'Agri-Renewable': 'Agritech',
    'B2b Agritech': 'Agritech',
    'Food And Agriculture Tech > Crop Tech': 'Agritech',
    
    # Catch-alls
    'Others': 'Other',
    'Unknown': 'Other',
}

mapping_clean = {normalize_sector(k): v for k, v in mapping.items()}
df['Sector'] = df['Sector'].replace(mapping_clean)
df['Sector'] = df['Sector'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

print(f"Sector cleaning: Unique before {df['Sector_Raw'].nunique()}, after {df['Sector'].nunique()}")

# === STEP 8: Final Dataset Assembly ===
# Select and order columns as per your request
final_columns = ['Company_Cleaned', 'Sector', 'Headquarters', 'Amount_Cr', 'Funding_Round_Type', 
                 'Lead_Investors', 'Year', 'Month']
final_df = df[final_columns].copy()

# Ensure no index issues
final_df = final_df.reset_index(drop=True)

# Save final cleaned dataset
final_df.to_csv('final_cleaned_dataset_1.csv', index=False)

print("\n=== FINAL CLEANING COMPLETE ===")
print(f"Final dataset: {len(final_df)} rows, columns: {list(final_df.columns)}")
print("\nFirst 5 rows:")
print(final_df.head().to_string(index=False))
print("\nSaved to 'final_cleaned_dataset_1.csv'")

Loaded original dataset: 7293 rows, 8 columns
Dropped exact duplicates: 0
Missing counts (Company, Year, Month):
Company    1
Year       0
Month      0
dtype: int64
Dropped rows with missing Company: 1

Basic cleaning done.
Company cleaning: Unique before 5404, after 5129


C:\Users\KEYUR\AppData\Local\Temp\ipykernel_10480\1922004385.py:184: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_money = df[hq_col].str.contains(money_pattern, case=False, na=False)
C:\Users\KEYUR\AppData\Local\Temp\ipykernel_10480\1922004385.py:194: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_temp_sorted[hq_col] = df_temp_sorted.groupby('Company')[hq_col].transform(lambda x: x.ffill().bfill())


Headquarters cleaning complete.
Unique HQs: 227
Amount_Cr stats: count    7.292000e+03
mean     1.746287e+04
std      5.408243e+05
min      0.000000e+00
25%      1.250000e+00
50%      1.245000e+01
75%      7.013750e+01
max      2.700000e+07
Name: Amount_Cr, dtype: float64
Funding_Round_Type cleaning: Unique values:
Funding_Round_Type
Seed                 2245
Private Equity       1361
Unknown              1261
Series A              533
Pre-Series A          440
Other                 330
Series B              263
Pre-Seed              182
Series C              162
Unspecified Round     137
Name: count, dtype: int64
Lead_Investors cleaning: Top unique values:
Lead_Investors
Unknown                      246
Unknown Investors             69
Inflection Point Ventures     63
Venture Catalysts             51
Sequoia Capital               34
Accel                         34
Indian Angel Network          33
Multiples                     32
Ratan Tata                    27
Tiger Global          

In [42]:
#  More cleaning on Company_Cleaned and Amount_Cr column

import pandas as pd
from rapidfuzz import fuzz, process
import re

# ============================================================
#  Load MASTER INPUT
# ============================================================
df = pd.read_csv("final_cleaned_dataset_1.csv")
df = df.reset_index(drop=True)


# ============================================================
#  FIXED VALUES (Original + NEW range 6262–6278)
# ============================================================
fixed_values = {
    # ORIGINAL
    5579: 1.1, 5587: 1.1, 5605: 36.1, 5606: 15.3, 5607: 32.8, 5615: 1400.0,
    5616: 513.5, 5620: 577.6, 5621: 18300.0, 5623: 550.0, 5625: 5000.0,
    5626: 2.2, 5627: 1300.0, 5641: 30.0, 5644: 29.0, 5645: 23.7, 5646: 84.2,
    5647: 431.6, 5649: 165.5, 5650: 240.2, 5660: 201.0, 5663: 728.1,
    5665: 0.485, 5667: 61.0, 5668: 32.4, 5669: 148.5, 5670: 129.5, 5671: 23.0,
    5674: 624.8, 5676: 11.7, 5677: 49.9, 5684: 37.0, 5686: 1.0, 5690: 12.3,
    5696: 76.0, 5720: 0.5, 5731: 0.16, 5736: 1.0, 5744: 0.5, 5755: 1.0,
    5767: 200.0, 5773: 0.4, 5791: 0.4, 5805: 3.0, 5823: 300.0, 5827: 0.2,
    5861: 1.2, 5874: 2.0, 5881: 215.0, 5906: 2.0, 5942: 0.4, 5951: 3.0,
    5994: 295.0, 6003: 0.5, 6008: 0.415, 6021: 24.9, 6083: 1.1, 6092: 50.0,
    6127: 3.3, 6159: 11.62, 6165: 2.0, 6184: 4.5, 6217: 5.4, 6221: 3.2,
    6239: 11.62, 6251: 19.92, 6300: 2.075, 6319: 2.1, 6325: 375.0, 6342: 2.0,
    6346: 226.0, 6370: 250.0, 6385: 8.3, 6387: 4.98, 6410: 2.905, 6417: 45.0,
    6440: 4.5, 6442: 224.1, 6448: 1.4193, 6454: 132.8, 6461: 15.77,
    6464: 22.659, 6472: 2.0584, 6486: 46.48, 6492: 4.15, 6495: 101.26,
    6505: 10.79, 6512: 41.5, 6516: 12.616, 6523: 6.5487, 6536: 66.4,
    6538: 107.9, 6541: 3.32, 6543: 41.5, 6571: 1.4027, 6594: 4.15,
    6611: 1.8841, 6659: 3.7931, 6702: 6.64, 6716: 3.7931, 6735: 1.9173,
    6757: 1.9256, 6774: 1.5023, 6797: 2.9133, 6818: 5.3369, 6828: 3.735,
    6870: 50.0, 6875: 24.9, 6878: 26.0, 6893: 157.7, 6895: 8.3, 6911: 16.6,
    6913: 99.6, 6991: 207.5, 7003: 2.49, 7017: 24.9, 7027: 30.0, 7051: 20.75,
    7085: 4.15, 7090: 5.0, 7094: 7.0, 7103: 41.5, 7174: 6.6, 7188: 6.75,
    7193: 22.41, 7215: 8.3, 7218: 3.0, 7268: 8.3, 7271: 4.0, 7291: 1.0,

    #  NEW VALUES
    6262: 5.81, 6263: 83.00, 6264: 166.00, 6265: 16.60, 6266: 9.13, 6267: 33.20,
    6268: 8.30, 6269: 13.28, 6270: 24.90, 6271: 6.64, 6272: 207.50, 6273: 29.05,
    6274: 224.10, 6275: 99.60, 6276: 74.70, 6277: 19.09, 6278: 9.96
}


# ============================================================
#  APPLY FIXED VALUES FIRST
# ============================================================
for idx, val in fixed_values.items():
    if idx < len(df):
        df.loc[idx, "Amount_Cr"] = val


# ============================================================
#  Auto-conversion for very large numbers
# ============================================================
if "Amount_Cr" not in df.columns:
    raise ValueError(" 'Amount_Cr' column not found!")

def convert_big_values(x):
    """
    If x > 9999 → treat as raw number → convert to Cr (INR assumed 83)
    Else return unchanged
    """
    try:
        num = float(x)
    except:
        return x
    
    if num > 9999:
        cr = (num * 83) / 10_000_000
        return round(cr, 2)
    return num

df["Amount_Cr"] = df["Amount_Cr"].apply(convert_big_values)


# ============================================================
#  REMOVE ROWS WHERE Amount_Cr == 0
# ============================================================
df = df[df["Amount_Cr"] != 0.0].reset_index(drop=True)


# ============================================================
#  CLEAN COMPANY NAMES (FUZZY MERGE > 80%)
# ============================================================
df["Company_Cleaned"] = (
    df["Company_Cleaned"]
    .astype(str)
    .str.lower()
    .str.strip()
)

companies = df["Company_Cleaned"].unique()
merged_count = 0

for i, company in enumerate(companies):
    match = process.extractOne(company, companies, scorer=fuzz.ratio)
    if match and match[1] > 80 and match[0] != company:
        df.loc[df["Company_Cleaned"] == company, "Company_Cleaned"] = match[0]
        merged_count += 1


# for Sectore column

# Normalize helper
def norm(x):
    return str(x).strip().lower() if pd.notna(x) else ""

# Final target buckets
def map_sector(raw):
    s = norm(raw)

    # FinTech / Financial
    if any(k in s for k in [
        "fin", "bank", "nbfc", "insurance", "wealth", "asset", "lending", "credit", "loan", "investment", "payments"
    ]):
        return "FinTech"

    # HealthTech
    if any(k in s for k in [
        "health", "med", "clinic", "pharma", "diagnos", "hospital", "femtech"
    ]):
        return "HealthTech"

    # EdTech
    if any(k in s for k in [
        "edu", "learn", "skill", "school", "campus"
    ]):
        return "EdTech"

    # AgriTech
    if any(k in s for k in [
        "agri", "farm", "crop", "soil"
    ]):
        return "AgriTech"

    # Food & Beverages / FoodTech
    if any(k in s for k in [
        "food", "bev", "kitchen", "nutrition", "beverage", "snack", "drink", "restaurant", "cook"
    ]):
        return "FoodTech"

    # E-commerce / Retail
    if any(k in s for k in [
        "ecom", "retail", "shopping", "marketplace", "d2c", "commerce", "mom & baby"
    ]):
        return "Ecommerce / Retail"

    # Consumer Internet
    if any(k in s for k in [
        "internet", "online", "consumer"
    ]):
        return "Consumer Internet"

    # Enterprise / SaaS / IT
    if any(k in s for k in [
        "saas", "enterprise", "b2b", "it", "software", "tech", "consulting", "productivity"
    ]):
        return "Enterprise / SaaS"

    # Artificial Intelligence
    if any(k in s for k in [
        "ai", "ml"
    ]):
        return "Artificial Intelligence"

    # DeepTech / Hardware / Semiconductors
    if any(k in s for k in [
        "deep", "quantum", "semicon", "chip", "3d", "robot", "hardware", "materials", "electronics"
    ]):
        return "DeepTech / Hardware"

    # Consumer Goods
    if any(k in s for k in [
        "fmcg", "goods", "cosmetic", "beauty", "personal care", "apparel", "fashion", "footwear"
    ]):
        return "Consumer Goods / D2C"

    # Logistics / Supply Chain
    if any(k in s for k in [
        "logist", "supply", "warehouse", "delivery", "last mile", "transport", "shipping"
    ]):
        return "Logistics / Supply Chain"

    # Mobility / Auto / EV
    if any(k in s for k in [
        "auto", "ev", "vehicle", "mobility", "bike", "tractor", "rental"
    ]):
        return "Mobility / Automotive / EV"

    # TravelTech / Hospitality
    if any(k in s for k in [
        "travel", "hotel", "hospitality", "hostel", "tour"
    ]):
        return "Travel / Hospitality"

    # Real Estate / PropTech
    if any(k in s for k in [
        "real", "prop", "housing", "construction", "interior", "rent"
    ]):
        return "Real Estate / PropTech"

    # Media & Entertainment / Gaming / Sports
    if any(k in s for k in [
        "media", "entertain", "game", "sport", "film", "music", "streaming", "video"
    ]):
        return "Media / Entertainment / Gaming"

    # Cybersecurity
    if any(k in s for k in [
        "cyber", "security", "defense", "defence"
    ]):
        return "Cybersecurity"

    # HR / Recruitment
    if any(k in s for k in [
        "hr", "recruit", "job", "workforce", "staff"
    ]):
        return "HRTech"

    # Marketing / Adtech
    if any(k in s for k in [
        "market", "advert", "brand", "loyalty"
    ]):
        return "Marketing / AdTech"

    # Climate / CleanTech / Energy
    if any(k in s for k in [
        "climate", "clean", "solar", "renew", "energy", "ev battery"
    ]):
        return "Climate / CleanTech / Energy"

    # SpaceTech
    if any(k in s for k in [
        "space", "satellite", "aero"
    ]):
        return "SpaceTech"

    # BioTech
    if any(k in s for k in [
        "bio", "genom", "life science"
    ]):
        return "BioTech"

    # Analytics
    if any(k in s for k in [
        "analytic", "data"
    ]):
        return "Analytics / Big Data"

    # LegalTech
    if "legal" in s:
        return "LegalTech"

    # Venture Capital
    if any(k in s for k in [
        "vc", "venture", "private equity", "fund"
    ]):
        return "Venture Capital"

    # Catch-all
    return "Others"


# Apply mapping
df["Sector_Cleaned"] = df["Sector"].apply(map_sector)

df["Sector_Cleaned"].value_counts()

df = df.rename(columns={
    "Sector": "Sub_Sector",
    "Sector_Cleaned": "Sector"
})



# for Headquarters
# Step 2: Updated mapping dict with standardizations (e.g., Bangalore/Bengaluru → Bangalore)
hq_mapping = {
    # Top Indian cities (standardized)
    'Bangalore': ('Bangalore', 'Karnataka', 'India'),
    'Bengaluru': ('Bangalore', 'Karnataka', 'India'),  # Standardized to Bangalore
    'Bengaluri': ('Bangalore', 'Karnataka', 'India'),  # Typo
    'Mumbai': ('Mumbai', 'Maharashtra', 'India'),
    'Andheri': ('Mumbai', 'Maharashtra', 'India'),  # Suburb
    'Thane': ('Mumbai', 'Maharashtra', 'India'),  # Suburb
    'Powai': ('Mumbai', 'Maharashtra', 'India'),  # Suburb
    'Chembur': ('Mumbai', 'Maharashtra', 'India'),  # Suburb
    'Gurgaon': ('Gurgaon', 'Haryana', 'India'),
    'Delhi': ('Delhi', 'Delhi', 'India'),
    'Noida': ('Delhi', 'Uttar Pradesh', 'India'),  # NCR → Delhi for city
    'Ghaziabad': ('Delhi', 'Uttar Pradesh', 'India'),  # NCR
    'Faridabad': ('Delhi', 'Haryana', 'India'),  # NCR
    'Chennai': ('Chennai', 'Tamil Nadu', 'India'),
    'Taramani': ('Chennai', 'Tamil Nadu', 'India'),  # Suburb
    'Pune': ('Pune', 'Maharashtra', 'India'),
    'Hyderabad': ('Hyderabad', 'Telangana', 'India'),
    'Hyderebad': ('Hyderabad', 'Telangana', 'India'),  # Typo
    'Noida': ('Noida', 'Uttar Pradesh', 'India'),  # Kept distinct but NCR-linked
    'Ahmedabad': ('Ahmedabad', 'Gujarat', 'India'),
    'Ahemadabad': ('Ahmedabad', 'Gujarat', 'India'),  # Typo
    'Jaipur': ('Jaipur', 'Rajasthan', 'India'),
    'Kolkata': ('Kolkata', 'West Bengal', 'India'),
    'Indore': ('Indore', 'Madhya Pradesh', 'India'),
    'Chandigarh': ('Chandigarh', 'Chandigarh', 'India'),
    'Surat': ('Surat', 'Gujarat', 'India'),
    'Coimbatore': ('Coimbatore', 'Tamil Nadu', 'India'),
    'Goa': ('Panaji', 'Goa', 'India'),
    'Panaji': ('Panaji', 'Goa', 'India'),
    'Lucknow': ('Lucknow', 'Uttar Pradesh', 'India'),
    'Kanpur': ('Kanpur', 'Uttar Pradesh', 'India'),
    'Vadodara': ('Vadodara', 'Gujarat', 'India'),
    'Kochi': ('Kochi', 'Kerala', 'India'),
    'Cochin': ('Kochi', 'Kerala', 'India'),  # Alias
    'Kochin': ('Kochi', 'Kerala', 'India'),  # Typo
    'Ernakulam': ('Kochi', 'Kerala', 'India'),  # District
    'Bhubaneswar': ('Bhubaneswar', 'Odisha', 'India'),
    'Bhubneswar': ('Bhubaneswar', 'Odisha', 'India'),  # Typo
    'Thiruvananthapuram': ('Thiruvananthapuram', 'Kerala', 'India'),
    'Jodhpur': ('Jodhpur', 'Rajasthan', 'India'),
    'Nashik': ('Nashik', 'Maharashtra', 'India'),
    'Raipur': ('Raipur', 'Chhattisgarh', 'India'),
    'Bhopal': ('Bhopal', 'Madhya Pradesh', 'India'),
    'Nagpur': ('Nagpur', 'Maharashtra', 'India'),
    'Dehradun': ('Dehradun', 'Uttarakhand', 'India'),
    'Kormangala': ('Bangalore', 'Karnataka', 'India'),  # Bangalore suburb (standardized)
    'Koramangala': ('Bangalore', 'Karnataka', 'India'),  # Corrected
    'Rourkela': ('Rourkela', 'Odisha', 'India'),
    'Secunderabad': ('Secunderabad', 'Telangana', 'India'),
    'Secundrabad': ('Secunderabad', 'Telangana', 'India'),  # Typo
    'Visakhapatnam': ('Visakhapatnam', 'Andhra Pradesh', 'India'),
    'Vishakhapatnam': ('Visakhapatnam', 'Andhra Pradesh', 'India'),  # Typo
    'Patna': ('Patna', 'Bihar', 'India'),
    'Udaipur': ('Udaipur', 'Rajasthan', 'India'),
    'Gwalior': ('Gwalior', 'Madhya Pradesh', 'India'),
    'Belgaum': ('Belgaum', 'Karnataka', 'India'),
    'Ranchi': ('Ranchi', 'Jharkhand', 'India'),
    'Rajkot': ('Rajkot', 'Gujarat', 'India'),
    'Roorkee': ('Roorkee', 'Uttarakhand', 'India'),
    'Tirunelveli': ('Tirunelveli', 'Tamil Nadu', 'India'),
    'Tiruchirappalli': ('Tiruchirappalli', 'Tamil Nadu', 'India'),
    'Tiruchirappall': ('Tiruchirappalli', 'Tamil Nadu', 'India'),  # Typo
    'Thanjavur': ('Thanjavur', 'Tamil Nadu', 'India'),
    'Varanasi': ('Varanasi', 'Uttar Pradesh', 'India'),
    'Udupi': ('Udupi', 'Karnataka', 'India'),
    'Satara': ('Satara', 'Maharashtra', 'India'),
    'Sonipat': ('Sonipat', 'Haryana', 'India'),  # Corrected from Sonepat
    'Panchkula': ('Panchkula', 'Haryana', 'India'),
    'Dewas': ('Dewas', 'Madhya Pradesh', 'India'),
    'Cuttack': ('Cuttack', 'Odisha', 'India'),
    'Gandhinagar': ('Gandhinagar', 'Gujarat', 'India'),
    'Gaya': ('Gaya', 'Bihar', 'India'),
    'Guwahati': ('Guwahati', 'Assam', 'India'),
    'Lonavala': ('Lonavala', 'Maharashtra', 'India'),
    'Mangalore': ('Mangalore', 'Karnataka', 'India'),
    'Mohali': ('Mohali', 'Punjab', 'India'),
    'Mysore': ('Mysore', 'Karnataka', 'India'),
    'Ludhiana': ('Ludhiana', 'Punjab', 'India'),
    'Haveri': ('Haveri', 'Karnataka', 'India'),
    'Hoshiarpur': ('Hoshiarpur', 'Punjab', 'India'),
    'Ichalkaranji': ('Ichalkaranji', 'Maharashtra', 'India'),
    'Jabalpur': ('Jabalpur', 'Madhya Pradesh', 'India'),
    'Jalandhar': ('Jalandhar', 'Punjab', 'India'),
    'Kota': ('Kota', 'Rajasthan', 'India'),
    'Kottayam': ('Kottayam', 'Kerala', 'India'),
    'Alappuzha': ('Alappuzha', 'Kerala', 'India'),
    'Amritsar': ('Amritsar', 'Punjab', 'India'),
    'Akola': ('Akola', 'Maharashtra', 'India'),
    'Ahmednagar': ('Ahmednagar', 'Maharashtra', 'India'),
    'Bahadurgarh': ('Bahadurgarh', 'Haryana', 'India'),
    
    # Indian states (no city change)
    'Maharashtra': (None, 'Maharashtra', 'India'),
    'Maharastra': (None, 'Maharashtra', 'India'),  # Typo
    'Karnataka': (None, 'Karnataka', 'India'),
    'Haryana': (None, 'Haryana', 'India'),
    'Tamil Nadu': (None, 'Tamil Nadu', 'India'),
    'Telangana': (None, 'Telangana', 'India'),
    'Telugana': (None, 'Telangana', 'India'),  # Typo
    'Uttar Pradesh': (None, 'Uttar Pradesh', 'India'),
    'Gujarat': (None, 'Gujarat', 'India'),
    'West Bengal': (None, 'West Bengal', 'India'),
    'Odisha': (None, 'Odisha', 'India'),
    'Orissia': (None, 'Odisha', 'India'),  # Typo
    'Bihar': (None, 'Bihar', 'India'),
    'Rajasthan': (None, 'Rajasthan', 'India'),
    'Rajastan': (None, 'Rajasthan', 'India'),  # Typo
    'Kerala': (None, 'Kerala', 'India'),
    'Madhya Pradesh': (None, 'Madhya Pradesh', 'India'),
    'Uttarakhand': (None, 'Uttarakhand', 'India'),
    
    # International (standardized similarly)
    'San Francisco': ('San Francisco', 'California', 'USA'),
    'Sanfrancisco': ('San Francisco', 'California', 'USA'),  # Typo
    'San Fracisco': ('San Francisco', 'California', 'USA'),  # Typo
    'Sfo': ('San Francisco', 'California', 'USA'),  # Abbrev
    'New York': ('New York', 'New York', 'USA'),
    'Newyork': ('New York', 'New York', 'USA'),  # Typo
    'Chicago': ('Chicago', 'Illinois', 'USA'),
    'Washington': ('Washington', 'District of Columbia', 'USA'),
    'Palo Alto': ('Palo Alto', 'California', 'USA'),
    'San Jose': ('San Jose', 'California', 'USA'),
    'Santa Clara': ('Santa Clara', 'California', 'USA'),
    'Seattle': ('Seattle', 'Washington', 'USA'),
    'Seatlle': ('Seattle', 'Washington', 'USA'),  # Typo
    'Dallas': ('Dallas', 'Texas', 'USA'),
    'Plano': ('Plano', 'Texas', 'USA'),
    'San Ramon': ('San Ramon', 'California', 'USA'),
    'Santa Monica': ('Santa Monica', 'California', 'USA'),
    'Boston': ('Boston', 'Massachusetts', 'USA'),
    'Austin': ('Austin', 'Texas', 'USA'),
    'Houston': ('Houston', 'Texas', 'USA'),
    'Indianapolis': ('Indianapolis', 'Indiana', 'USA'),
    'Irvine': ('Irvine', 'California', 'USA'),
    'Newark': ('Newark', 'New Jersey', 'USA'),
    'Oakland': ('Oakland', 'California', 'USA'),
    'Parsippany': ('Parsippany', 'New Jersey', 'USA'),
    'Stamford': ('Stamford', 'Connecticut', 'USA'),
    'Troy': ('Troy', 'Michigan', 'USA'),
    'Wilmington': ('Wilmington', 'Delaware', 'USA'),
    'Dover': ('Dover', 'Delaware', 'USA'),
    'Petaluma': ('Petaluma', 'California', 'USA'),
    'Burnsville': ('Burnsville', 'Minnesota', 'USA'),
    'Cumberland Furnace': ('Cumberland Furnace', 'Tennessee', 'USA'),
    'Linthicum Heights': ('Linthicum Heights', 'Maryland', 'USA'),  # Fixed
    'Menlo Park': ('Menlo Park', 'California', 'USA'),
    'St. Louis': ('St. Louis', 'Missouri', 'USA'),  # For Missourie
    'Maryland': (None, 'Maryland', 'USA'),
    'Massachusetts': (None, 'Massachusetts', 'USA'),
    'Utah': (None, 'Utah', 'USA'),
    'Delaware': (None, 'Delaware', 'USA'),
    'Wyoming': (None, 'Wyoming', 'USA'),
    'Us': (None, None, 'USA'),
    'Usa': (None, None, 'USA'),
    'United States': (None, None, 'USA'),
    'India': (None, None, 'India'),
    
    # Other international (no changes needed)
    'Singapore': (None, None, 'Singapore'),
    'London': ('London', None, 'United Kingdom'),
    'United Kingdom': (None, None, 'United Kingdom'),
    'Sydney': ('Sydney', 'New South Wales', 'Australia'),
    'Berlin': ('Berlin', None, 'Germany'),
    'Paris': ('Paris', None, 'France'),
    'France': (None, None, 'France'),
    'Tokyo': ('Tokyo', None, 'Japan'),
    'Seoul': ('Seoul', None, 'South Korea'),
    'Shanghai': ('Shanghai', None, 'China'),
    'Beijing': ('Beijing', None, 'China'),
    'Dubai': ('Dubai', None, 'UAE'),
    'Abu Dhabi': ('Abu Dhabi', None, 'UAE'),
    'Riyadh': ('Riyadh', None, 'Saudi Arabia'),
    'Tangerang': ('Tangerang', 'Banten', 'Indonesia'),
    'Jiaxing': ('Jiaxing', 'Zhejiang', 'China'),
    'Karachi': ('Karachi', 'Sindh', 'Pakistan'),
    'Bangkok': ('Bangkok', None, 'Thailand'),
    'Auckland': ('Auckland', None, 'New Zealand'),
    'Nairobi': ('Nairobi', None, 'Kenya'),
    'Milan': ('Milan', 'Lombardy', 'Italy'),  # From Milano
    'Milano': ('Milan', 'Lombardy', 'Italy'),
    'Newcastle Upon Tyne': ('Newcastle upon Tyne', 'Tyne and Wear', 'United Kingdom'),
    'Israel': (None, None, 'Israel'),
    'Bangladesh': (None, None, 'Bangladesh'),  # From Bangaldesh
    
    # Misc
    'Unknown': (None, None, None),
    'Nan': (None, None, None),
    'Small Towns': (None, None, None),
    'Retail': (None, None, None),
    'Food Beverages': (None, None, None),
    'Eximpe Is Managing Sellers': (None, None, None),
    'Rajsamand': ('Rajsamand', 'Rajasthan', 'India'),
    'Samastipur': ('Samastipur', 'Bihar', 'India'),
    'Samsitpur': ('Samastipur', 'Bihar', 'India'),  # Typo
    'Posha': (None, None, None),
    'The Nilgiris': (None, 'The Nilgiris', 'India'),
    'Vitznau': ('Vitznau', 'Lucerne', 'Switzerland'),
    'Silvassa': ('Silvassa', 'Dadra and Nagar Haveli', 'India'),
    # Add more if new uniques appear
}

# Step 3: Function to apply mapping
def map_hq(value):
    if pd.isna(value):
        return (None, None, None)
    clean_val = str(value).strip()
    return hq_mapping.get(clean_val, (None, None, None))

# Step 4: Apply to create new columns
df[['city', 'state', 'country']] = df['Headquarters'].apply(map_hq).apply(pd.Series)

# Step 5: Verify (now Bangalore should consolidate ~1758 + any Bengaluru)
print("Original Headquarters Counts (top 10):")
print(df['Headquarters'].value_counts().head(10))

print("\nUpdated City Counts (top 10):")
print(df['city'].value_counts().head(10))  # Bangalore should be higher

print("\nState Counts (top 10):")
print(df['state'].value_counts().head(10))

print("\nCountry Counts:")
print(df['country'].value_counts())


# ============================================================
#  SAVE FINAL OUTPUT
# ============================================================
df.loc[4228, "Amount_Cr"] = 15.0
df.to_csv("final_cleaned_dataset_2.csv", index=False)

print("\n FINAL CLEANING COMPLETE ")
print(" Saved → final_cleaned_dataset_2.csv")
print(" Company merges:", merged_count)
print(" Remaining rows:", len(df))


Original Headquarters Counts (top 10):
Headquarters
Bangalore    1758
Mumbai        985
Gurgaon       684
Delhi         637
Unknown       239
Chennai       235
Pune          205
Hyderabad     185
Noida         165
Ahmedabad      70
Name: count, dtype: int64

Updated City Counts (top 10):
city
Bangalore    1762
Mumbai       1009
Gurgaon       684
Delhi         649
Chennai       237
Pune          205
Hyderabad     186
Noida         165
Ahmedabad      72
Jaipur         52
Name: count, dtype: int64

State Counts (top 10):
state
Karnataka        1799
Maharashtra      1262
Haryana           727
Delhi             637
Tamil Nadu        260
Telangana         199
Uttar Pradesh     191
Gujarat           107
Rajasthan          64
West Bengal        43
Name: count, dtype: int64

Country Counts:
country
India             5424
USA                100
Singapore           22
United Kingdom       8
China                3
UAE                  2
Australia            2
France               2
Italy          

In [5]:
import pandas as pd
import numpy as np
from datetime import datetime
import random

# -----------------------------
# Month Mapping
# -----------------------------
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}

# -----------------------------
# LOAD MAIN DATASET
# -----------------------------
print("Loading and cleaning data...")
df = pd.read_csv("final_cleaned_dataset_2.csv")

df.columns = df.columns.str.strip()
df["Amount_Cr"] = pd.to_numeric(df["Amount_Cr"], errors="coerce").fillna(0)

# Guard: Create Company_Cleaned if missing
if "Company_Cleaned" not in df.columns:
    df["Company_Cleaned"] = df.get("Company", "")

# -----------------------------
# Create Month_Num and Date
# -----------------------------
df["Month_Num"] = df["Month"].map(month_map).fillna(0).astype(int)
df["Day"] = [random.randint(1, 28) for _ in range(len(df))]

tmp = df[["Year", "Month_Num", "Day"]].rename(columns={"Year": "year", "Month_Num": "month", "Day": "day"})
df["Date"] = pd.to_datetime(tmp, errors="coerce")

# Remove invalid rows
df = df.dropna(subset=["Company_Cleaned", "Sector", "Headquarters"])

# -----------------------------
# Add Seasonality Features
# -----------------------------
df["Quarter"] = df["Month_Num"].apply(lambda m: (m - 1) // 3 + 1)
df["FY"] = np.where(df["Month_Num"] >= 4, df["Year"], df["Year"] - 1)

print("Loaded rows:", len(df))

# =============================================================
# STEP 1: Company Funding History (cumulative prior)
# =============================================================
print("\nComputing Cumulative Funding Prior...")

df = df.sort_values(["Company_Cleaned", "Date"], kind="mergesort")
df["Cumulative_Funding_Prior"] = (
    df.groupby("Company_Cleaned")["Amount_Cr"]
      .transform(lambda s: s.cumsum().shift(1).fillna(0))
)

# =============================================================
# STEP 2: LOCAL MARKET SIGNALS (Rolling Windows)
# =============================================================
print("\nComputing Local Market Signals...")

df["HQ_Sector"] = df["Headquarters"].fillna("Unknown") + "_" + df["Sector"]

# Save original index
orig_idx = df.index
df = df.sort_values(["HQ_Sector", "Date"], kind="mergesort")

# Rolling function
def rolling_on_date(g, col, window, agg_func):
    if g.empty:
        return pd.Series([0] * len(g), index=g.index)

    g2 = g.set_index("Date")

    rolled = (
        g2[col]
        .rolling(window=window, min_periods=1)
        .agg(agg_func)
        .shift(1)
        .fillna(0)
    )

    # FIX: return aligned series without reindex duplicate errors
    rolled = rolled.reset_index(drop=True)

    return pd.Series(rolled.values, index=g.index)


# ---------- FIXED GROUPBY APPLY (FULL DF, NOT SERIES) ----------
df["Rolling_6m_Funding_Lagged"] = (
    df.groupby("HQ_Sector", group_keys=False)
      .apply(lambda g: rolling_on_date(g, "Amount_Cr", "180D", "sum"))
)

df["Rolling_6m_Rounds_Lagged"] = (
    df.groupby("HQ_Sector", group_keys=False)
      .apply(lambda g: rolling_on_date(g, "Amount_Cr", "180D", "count"))
)

df["Rolling_6m_Median_Size_Lagged"] = (
    df.groupby("HQ_Sector", group_keys=False)
      .apply(lambda g: rolling_on_date(g, "Amount_Cr", "180D", "median"))
)

df["Rolling_12m_Funding_Lagged"] = (
    df.groupby("HQ_Sector", group_keys=False)
      .apply(lambda g: rolling_on_date(g, "Amount_Cr", "365D", "sum"))
)

df["Rolling_12m_Rounds_Lagged"] = (
    df.groupby("HQ_Sector", group_keys=False)
      .apply(lambda g: rolling_on_date(g, "Amount_Cr", "365D", "count"))
)

df["Rolling_12m_Median_Size_Lagged"] = (
    df.groupby("HQ_Sector", group_keys=False)
      .apply(lambda g: rolling_on_date(g, "Amount_Cr", "365D", "median"))
)

# Restore order
df = df.reindex(orig_idx)

# Cleanup
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].round(2)

# df.to_csv("df_enhanced_fixed.csv", index=False)
print("Saved df_enhanced_fixed.csv")

# =============================================================
# STEP 3: COMPANY-LEVEL AGGREGATION
# =============================================================
print("\nBuilding Company-Level Aggregates...")

company_df = df.groupby("Company_Cleaned").agg({
    "Amount_Cr": ["sum", "max", "count"],
    "Sub_Sector": "first",
    "Headquarters": "first",
    "city": "first",
    "state": "first",
    "country": "first",
    "Funding_Round_Type": lambda s: ", ".join(s.value_counts().head(3).index),
    "Year": lambda s: ", ".join(map(str, sorted(s.unique()))),
    "Lead_Investors": lambda s: ", ".join(s.astype(str).unique()),
    "Cumulative_Funding_Prior": "last",
    "Date": ["min", "max"],
    "Quarter": lambda s: ", ".join(map(str, sorted(s.unique()))),
    "FY": lambda s: ", ".join(map(str, sorted(s.unique()))),
}).reset_index()

company_df.columns = [
    "Company_Cleaned", "Total_Funding", "Max_Funding", "Num_Rounds",
    "Sub_Sector", "Headquarters", "City", "State", "Country",
    "Top_Round_Types", "Years", "Investors",
    "Last_Cumulative_Prior", "First_Date", "Last_Date", "Quarters", "FYs"
]

company_df["First_Year"] = pd.to_datetime(company_df["First_Date"]).dt.year
company_df["Last_Year"] = pd.to_datetime(company_df["Last_Date"]).dt.year

company_df = company_df.sort_values("Total_Funding", ascending=False)
# company_df.to_csv("company_enhanced_fixed.csv", index=False)

print("Saved company_enhanced_fixed.csv")

# =============================================================
# STEP 4: INVESTOR-LEVEL AGGREGATION
# =============================================================
print("\nBuilding Investor-Level Aggregates...")

df["Lead_Investors_List"] = df["Lead_Investors"].astype(str).str.split(", ")
inv_exp = df.explode("Lead_Investors_List").dropna(subset=["Lead_Investors_List"])

company_investor = inv_exp.groupby(["Company_Cleaned", "Lead_Investors_List"])["Amount_Cr"].sum().reset_index()
company_investor["Share"] = company_investor.groupby("Company_Cleaned")["Amount_Cr"].transform(lambda s: s / s.sum())
company_investor["HHI_Contribution"] = company_investor["Share"] ** 2

company_hhi = company_investor.groupby("Company_Cleaned")["HHI_Contribution"].sum().reset_index()
company_hhi["Unique_Lead_Count"] = company_investor.groupby("Company_Cleaned")["Lead_Investors_List"].nunique().values
company_hhi.columns = ["Company_Cleaned", "HHI", "Unique_Lead_Count"]

df = df.merge(company_hhi, on="Company_Cleaned", how="left")

investor_df = df.explode("Lead_Investors_List").dropna(subset=["Lead_Investors_List"])

agg = {
    "Amount_Cr": "sum",
    "Company_Cleaned": "nunique",
    "Date": "max",
    "Sub_Sector": lambda s: ", ".join(s.value_counts().head(3).index),
    "Funding_Round_Type": lambda s: ", ".join(s.value_counts().head(3).index),
    "Headquarters": lambda s: ", ".join(s.value_counts().head(3).index),
    "HHI": "mean",
    "Unique_Lead_Count": "mean"
}

investor_df = investor_df.groupby("Lead_Investors_List").agg(agg).reset_index()
investor_df.columns = [
    "Investor", "Total_Invested", "Num_Investments", "Last_Date",
    "Top_Sub_Sectors", "Top_Stages", "Top_Cities",
    "Avg_HHI", "Avg_Unique_Leads"
]

investor_df["Last_Investment"] = pd.to_datetime(investor_df["Last_Date"])
investor_df = investor_df.sort_values("Total_Invested", ascending=False)

# investor_df.to_csv("investor_enhanced_fixed.csv", index=False)

print("Saved investor_enhanced_fixed.csv")

# =============================================================
# STEP 5: MARKET SIGNALS SUMMARY
# =============================================================
print("\nBuilding Market Signals Summary...")

market = df.groupby(["Headquarters", "Sector"]).agg({
    "Rolling_6m_Funding_Lagged": "mean",
    "Rolling_6m_Rounds_Lagged": "mean",
    "Rolling_6m_Median_Size_Lagged": "mean",
    "Rolling_12m_Funding_Lagged": "mean",
    "Rolling_12m_Rounds_Lagged": "mean",
    "Rolling_12m_Median_Size_Lagged": "mean",
    "Amount_Cr": "sum",
    "Date": "count"
}).reset_index()

market.columns = [
    "Headquarters", "Sector",
    "Avg_6m_Funding", "Avg_6m_Rounds", "Avg_6m_Median_Size",
    "Avg_12m_Funding", "Avg_12m_Rounds", "Avg_12m_Median_Size",
    "Total_Funding", "Total_Rounds"
]

# market.to_csv("market_signals_summary_fixed.csv", index=False)
print("Saved market_signals_summary_fixed.csv")

print("\n🎉 ALL FILES PROCESSED SUCCESSFULLY!")


Loading and cleaning data...
Loaded rows: 5849

Computing Cumulative Funding Prior...

Computing Local Market Signals...
Saved df_enhanced_fixed.csv

Building Company-Level Aggregates...
Saved company_enhanced_fixed.csv

Building Investor-Level Aggregates...
Saved investor_enhanced_fixed.csv

Building Market Signals Summary...
Saved market_signals_summary_fixed.csv

🎉 ALL FILES PROCESSED SUCCESSFULLY!
